# PoC: NBNL Equipment 3.0 in SHACL

## Architecture

* A profile is a profile in the sense of the W3C PROF vocabulary.
    * It is also a SHACL shapes graph, inspired by the upcoming work in SHACL 1.2 Profiling (section 4).
    * Additionally, ADMS is used for more fine-grained distinction between logical and physical representation. We may consider being compliant with EU SEMiC's ADMS-AP.

### IRIs

TODO.

### Metadata

TODO.

## Modelling

* The profile source is maintained in YAML-LD. This single file is the entire profile. QUESTION: If there is a profile-bound vocabulary, this statement is not true. But more likely we will introduce semantic extensions in the profile group-bound CIM NL vocabulary, so then this statement would make sense. This also makes everything a lot easier.
    * Supporting profile resources are pointed to, but do not make 
    * Note that YAML-LD is not widely supported yet and still in draft. If this turns out to be an issue we can still decide to just use JSON-LD instead.
* It is standard SHACL so SHACL modelling tools can be used.
* Source files are kept as DRY as possible for improved maintenance.
    * Imports or other means of dependency declaration are utilized in standardized ways. For example, one could import the entire canonical CIM vocabulary here. This will help the reasoner and add labels and descriptions. On top of that, this allows for profiling/autocompleting from a wider vocabulary then what is currently in use. This is precisely what you want during modelling.
        * Note that for validation these imports will not do anything for you, since they are not added to the data graph. But for modelling tools they can help as described just now.
* There is a shared NBNL Profile JSON-LD context file which is imported in the context. This makes authoring easier and more uniform.
* In the future, own tooling needs to be improved as well.
    * For example a VS Code extension which autocompletes from given ontologies.
* THOUGHT: Pre-processing using an OWL vocabulary which infers that alles objects of triples with predicate `nl:hasClassShape` are of types `rdfs:Class` and `sh:NodeShape`, and more such reasoning to make authoring easier. This didn't seem to work, and I also worry about tooling interop here. Lastly, people won't be writing much by hand anyway, and even if they do: how much of a big deal is writing such type statements?

### Profile source file

In [1]:
%cat nbnl-equipment.prof.shacl.yaml

'@context':
  '@version': 1.1
  '@import': http://localhost:8000/remote/nbnl-prof.context.jsonld
  cim: https://cim.ucaiug.io/ns#
'@id': https://modellen.netbeheernederland.nl/nbnl-equipment/3.0
entailment: http://www.w3.org/ns/entailment/RDFS
isVersionOf: https://modellen.netbeheernederland.nl/nbnl-equipment
version: '3.0'
'@type':
  - prof:Profile
  - sh:ShapesGraph
  - owl:Ontology
imports:
  - remote/nbnl-prof.owl.jsonld
  - nbnl-equipment.rdfs.jsonld  # TODO: Should be provided by data graph (or indirectly by some ontology)!
  - remote/base-voltage.shacl.jsonld
title: NBNL Equipment Profile
description: >-
  CIM profile for modelling equipment.
conformsTo:
  - https://cim17.org
  # - https://eu-dcat-ap?
landingPage: https://modellen.netbeheernederland.nl/nbnl-equipment/3.0
issued: 2026-09-11
contactPoint:
  fullName: Bart Kleijngeld
  email: bart.kleijngeld@alliander.com
publisher:
  name: Netbeheer Nederland
language: EN  #TODO
type: application-profile  # TODO: Use a nested cont

## Building

* A built profile is a JSON-LD document which encodes a RDF dataset:
    * Default graph contains the profile and its metadata.
    * Optionally there is a named graph for the profile vocabulary which contains precisely (and no more) the vocabulary needed to interpret (and validate) the profile. This improves performance and makes the architecture less brittle because no dependencies on external files remain.
* Imports are materialized, and only the subset that is used in the profile (i.e. constraints are defined for them) is selected.
    * CHORE: Builds lead to deterministic artifacts, which is nice on the one hand but it does mean provision metadata should be provided so that at all times it can be known what version of resources were used to buid.
    * ISSUE: Furthermore, however, the build process is not necessarily deterministic. If for example CIM updates and the import fetches those changes, the build process will materialize them. You _will_ be able to tell because of the aforementioned metadata, but still, the building process is not entirely idempotent.

![Overview of the building process](img/building-process.excalidraw.svg)

## Validating

* > "The data graph is expected to include all the ontology axioms related to the data and especially all the `rdfs:subClassOf` triples in order for SHACL to correctly identify class targets and validate Core SHACL constraints."
* Entailment cannot be relied upon from each validator implementation, so best is to reason over the graphs beforehand and materialize the inferred triples.